# 01 · Tensors and autograd: build the engine that trains your models

*Base Model Training*, notebook 1 of 4

A base model starts from random weights and learns everything it knows from data, one gradient step at a time. Frameworks such as PyTorch hide the machinery that computes those gradients. In this notebook you build it yourself: a small `Tensor` class that records every operation on the way forward and then runs the chain rule backwards through that record. The rest of this course, and the later deep-learning courses, train every model with the engine you finish here.

## What you will build

- `count_parameters(layer_sizes)`: how many numbers a network learns;
- `unbroadcast(grad, shape)`: the shape repair that NumPy broadcasting makes necessary;
- `topological_order(root)` and `backward_pass(root, grad)`: reverse-mode differentiation, including gradient accumulation;
- `tensor_mul`: your first complete operation, forward and backward;
- the backward rules `matmul_backward`, `relu_backward` and `sum_backward`, and `tensor_mean`, built from operations you already have;
- `log_softmax_forward` and `log_softmax_backward`: the numerically stable heart of classification losses;
- and, to put it all together, a 2 → 16 → 2 network that your engine trains.

## Learning objectives

By the end of this notebook you will be able to:

1. separate a model's architecture, weights and training objective, and count its parameters from its layer sizes;
2. follow the shape of a batch through a network, and explain broadcasting;
3. explain how reverse-mode automatic differentiation records a graph and walks it backwards;
4. write forward and backward rules for elementwise, matrix and reduction operations, and check each one against a numerical gradient;
5. explain why gradients accumulate, and why a numerically stable log-softmax matters.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you type a function yourself, then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy does every calculation, and the standard-library `time` module measures one speed comparison. Matplotlib is not needed in this notebook. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded, and every random number comes from a seeded generator.
- You should be comfortable with NumPy arrays, matrix multiplication and the chain rule, as taught in *Neural Networks from Scratch*. That course derived the gradients of one fixed network by hand; here you build a tool that derives them for any network you can write down.

## Setup

Run this cell first. It imports NumPy and `time`, creates a seeded random generator and defines `practice`, the helper that checks each function you write against its reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

Some pieces you will write do their work by changing objects (they fill in `.grad`), and one has more than one correct answer: a graph can usually be visited in several valid orders. `practice` compares return values, so for those pieces the check uses a **probe**, a small function that runs your code on a fresh example and returns plain numbers to compare. `practice_via` wraps that up and, like `practice`, hands back your function once it matches.

In [ ]:
def practice_via(name, yours, reference, probe, cases, atol=1e-6):
    """Like practice, but compares probe(function, *case) for your function and for the reference.

    Use it when a function works by changing objects (filling in .grad, updating
    weights) or when several answers are correct: the probe runs the function on
    a fresh example and turns the outcome into plain numbers. Returns your
    function once the measurements match, otherwise the reference.
    """
    def measure_yours(*case):
        return probe(yours, *case)

    def measure_reference(*case):
        return probe(reference, *case)

    chosen = practice(name, measure_yours, measure_reference, cases, atol)
    return yours if chosen is measure_yours else reference

## 1 · Architecture, weights and objective

A trained model is three separate things:

- the **architecture**: which operations run, in which order, on which shapes. A 2 → 16 → 2 multilayer perceptron (MLP) multiplies its 2 inputs by a weight matrix, adds a bias, applies ReLU, and repeats with a second layer that produces 2 outputs;
- the **weights**, or parameters: the numbers inside those matrices and biases, which training changes;
- the **objective**: the single number that training tries to reduce, such as the average cross-entropy between predicted class probabilities and true labels.

Two networks with the same architecture behave differently when they learned from different data or objectives. **Training from scratch** starts the weights at random values; **fine-tuning** starts from weights that someone has already trained and adapts them. "Base model" names that first stage of training; it says nothing about how capable the model is.

Before choosing layer sizes, write a contract. In this course the contract reads: *input*, two measured coordinates; *output*, two class scores (logits); *target*, which side of a nonlinear boundary the point lies on; *baseline*, always predict the most common training class; *acceptance*, beat the baseline on validation data, then measure the chosen model once on held-out test data.

### Shapes through one batch

For a batch of $n$ examples, a dense layer computes $Z = XW + b$ with

$$X: (n, n_\text{in}), \qquad W: (n_\text{in}, n_\text{out}), \qquad b: (n_\text{out},), \qquad Z: (n, n_\text{out}).$$

Frameworks such as PyTorch store the transposed matrix, of shape $(n_\text{out}, n_\text{in})$, and compute $XW^\top + b$; the arithmetic is the same. The batch size $n$ never appears in a weight's shape, which is why one set of weights serves batches of any size.

In [ ]:
demo_rng = np.random.default_rng(1)
X_demo = demo_rng.normal(size=(8, 2))                     # a batch: 8 examples, 2 features each
W1_demo, b1_demo = demo_rng.normal(0.0, 1.0, (2, 16)), np.zeros(16)
W2_demo, b2_demo = demo_rng.normal(0.0, 0.35, (16, 2)), np.zeros(2)
H_demo = np.maximum(X_demo @ W1_demo + b1_demo, 0.0)      # hidden activations after ReLU
logits_demo = H_demo @ W2_demo + b2_demo
for name, array in [("X", X_demo), ("W1", W1_demo), ("b1", b1_demo), ("H", H_demo),
                    ("W2", W2_demo), ("b2", b2_demo), ("logits", logits_demo)]:
    print(f"{name:>6}: shape {str(array.shape):8} dtype {array.dtype}")

### ✍️ Your turn: `count_parameters`

Write `count_parameters(layer_sizes)`:

- **input:** a list of layer widths, inputs first, such as `[2, 16, 2]`;
- **output:** the number of trainable numbers, as an `int`. Each consecutive pair $(n_\text{in}, n_\text{out})$ contributes $n_\text{in} \times n_\text{out}$ weights plus $n_\text{out}$ biases.

Hint: `zip(layer_sizes[:-1], layer_sizes[1:])` walks through the consecutive pairs.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def count_parameters_reference(layer_sizes):
    return sum(n_in * n_out + n_out for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]))

In [ ]:
def count_parameters(layer_sizes):
    """Weights plus biases of a dense network with these widths, e.g. [2, 16, 2] -> 82."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
count_parameters = practice("count_parameters", count_parameters, count_parameters_reference, cases=[
    ([2, 16, 2],),
    ([2, 32, 2],),
    ([4, 8, 8, 3],),
    ([5, 1],),          # edge case: a single layer with one output
])

**Expected behaviour.** The 2 → 16 → 2 network has $2 \times 16 + 16 = 48$ parameters in its first layer and $16 \times 2 + 2 = 34$ in its second: 82 in total. Doubling the hidden width to 32 gives 162, roughly twice as many, because both layers grow. The batch of 8 examples above appears nowhere: the batch size changes how many examples are processed at once, not how many numbers are learned.

### 🧪 Try it: change the width, then break the shapes

Predict `count_parameters([2, hidden, 2])` before you run the cell, then change `hidden` and run it again. The second half feeds the network three features instead of two. Read the error message: it names the two inner dimensions that fail to match.

In [ ]:
hidden = 16  # 🧪 try 32 and 64, predicting the count first
print(f"[2, {hidden}, 2] has {count_parameters([2, hidden, 2])} parameters")

X_three = demo_rng.normal(size=(8, 3))  # three features, but W1 expects two
try:
    X_three @ W1_demo
except ValueError as error:
    print("Shape error:", error)

### Numbers have types as well as shapes

Network inputs and weights are floating-point numbers; this engine stores everything as `float64`. Class targets are **integer** indices, one per example, used to pick an entry out of each row of scores. A **boolean** mask (keep or drop) is a different object from a floating-point mask that is added to scores. A correctly shaped array can still hold the wrong kind of number, so inspect `shape`, `dtype` and a few values together. Frameworks such as PyTorch also track a *device*, CPU or GPU memory, for every tensor, and every tensor in one operation must be on the same device. NumPy arrays always live in main memory, which keeps this course on a laptop's CPU.

In [ ]:
targets_demo = np.array([0, 1, 1, 0])                # class indices: integers
keep_demo = np.array([True, False, True, True])      # a boolean mask
print("dtypes:", targets_demo.dtype, keep_demo.dtype, X_demo.dtype)
print("score of the first example for its target class:", logits_demo[0, targets_demo[0]].round(4))
print("integer division floors without warning:", np.array([1, 2, 3]) // 2)

## 2 · Why build automatic differentiation?

Training needs $\partial L / \partial \theta$ for every parameter $\theta$. A **numerical gradient** nudges one parameter at a time,

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \varepsilon) - L(\theta_i - \varepsilon)}{2\varepsilon},$$

which costs two forward passes *per parameter*: 164 passes for the 82 parameters above, and billions for a large model. **Reverse-mode automatic differentiation** obtains every partial derivative in one backward sweep, which costs a small multiple of one forward pass.

The idea is simple. During the forward pass, every operation records its inputs (its *parents*) and a rule for sending a gradient back to them. If $y = f(x)$ and a later loss $L$ depends on $y$, the chain rule gives

$$\bar{x} = \bar{y} \, \frac{\partial y}{\partial x}, \qquad \text{where } \bar{y} = \frac{\partial L}{\partial y} \text{ has the same shape as } y.$$

So each operation needs only a **local** rule: given $\bar y$, the *upstream gradient*, return one gradient per parent, each with that parent's shape. When a value feeds several later operations, its gradient is the **sum** of what flows back along every path. The backward pass visits the operations in reverse order and applies their rules.

## 3 · The `Tensor` skeleton

A `Tensor` wraps a NumPy array and remembers how it was made:

| attribute | meaning |
|---|---|
| `data` | the values, a `float64` array |
| `requires_grad` | `True` for weights, and for anything computed from them |
| `grad` | filled in by the backward pass: $\partial L / \partial\,$`data`, with the same shape as `data` |
| `_parents` | the tensors this one was computed from |
| `_backward` | the local rule: upstream gradient → one gradient per parent |
| `op` | a name for printing, such as `"add"` |

`_result` creates the output of an operation. If no parent needs a gradient, it returns a plain tensor with no history, so constants cost nothing. `lift` turns a number or an array into a `Tensor`, so that `x + 1.0` works. `backward()` hands over to `backward_pass`, which you write in section 5. Read the class, then run it; it does nothing interesting yet.

In [ ]:
class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    def backward(self, grad=None):
        backward_pass(self, grad)  # you write backward_pass in section 5

Addition is the simplest operation. For $y = a + b$, both local derivatives are 1, so the rule passes the upstream gradient $\bar y$ unchanged to both parents. The function below builds the output with `_result`, and the last line attaches it to the class as `+`.

In [ ]:
def tensor_add(self, other):
    other = Tensor.lift(other)
    return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")


Tensor.__add__ = Tensor.__radd__ = tensor_add

a = Tensor([1.0, 2.0, 3.0], requires_grad=True)
c = a + 10.0
print(c, c.data)
print("parents:", c._parents)
print("the local rule sends", c._backward(np.ones(3)), "back to (a, 10.0)")

### Tools for checking gradients

`numerical_grad(f, tensor)` computes central differences for every entry of `tensor.data`, where `f()` rebuilds a scalar loss each time it is called. `relative_error(a, b)` compares two arrays on a scale-free basis: below about $10^{-6}$ a backward rule is right, while a wrong rule usually gives something near 1.

`gradient_check(op, *arrays)` combines them for one operation. It runs `op` on fresh tensors, sends a fixed, irregular upstream gradient $U$ back with `backward()`, and compares each input's `.grad` with the numerical gradient of $\sum U \odot \text{op}(\ldots)$. An irregular $U$, rather than all ones, catches rules that mix up positions.

In [ ]:
def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))


def gradient_check(op, *arrays, **kwargs):
    """Worst relative error between backward() and central differences for op(*tensors, **kwargs)."""
    leaves = [Tensor(np.array(array, dtype=float), requires_grad=True) for array in arrays]
    out = op(*leaves, **kwargs)
    upstream = np.cos(np.arange(out.data.size) + 1.0).reshape(out.shape)
    out.backward(upstream)

    def weighted_sum():
        return Tensor((op(*leaves, **kwargs).data * upstream).sum())

    return max(relative_error(leaf.grad, numerical_grad(weighted_sum, leaf)) for leaf in leaves)

## 4 · Broadcasting, and undoing it in the backward pass

NumPy **broadcasts**: in `X + b`, with `X` of shape `(2, 3)` and `b` of shape `(3,)`, the bias is reused for every row. Shapes are compared from the right, and a missing leading axis, or an axis of size 1, is stretched to match.

The rule for addition returns the upstream gradient unchanged, with shape `(2, 3)`. But `b.grad` must have `b`'s shape, `(3,)`. Each entry of `b` was used once per row, so its gradient is the **sum** of the upstream gradient over the rows. In general, to bring a gradient back to a parent's shape:

1. sum away the leading axes that broadcasting added;
2. for every axis where the parent has size 1 but the gradient does not, sum over that axis and keep it, with `keepdims=True`.

In [ ]:
x_small = np.arange(6.0).reshape(2, 3)
bias_small = np.array([10.0, 20.0, 30.0])
print("x + bias:\n", x_small + bias_small)
upstream_small = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("bias was used in both rows, so its gradient is the column sum:", upstream_small.sum(axis=0))

### ✍️ Your turn: `unbroadcast`

Write `unbroadcast(grad, shape)`:

- **inputs:** `grad`, an array with the broadcast shape, such as `(2, 3)`; `shape`, the parent's shape as a tuple, such as `(3,)`, `(1, 3)` or `()`;
- **output:** an array of exactly `shape`, holding `grad` summed over every broadcast axis.

Hints: a `while grad.ndim > len(shape):` loop with `grad = grad.sum(axis=0)` handles rule 1. Then `for axis, size in enumerate(shape):` with `grad.sum(axis=axis, keepdims=True)` handles rule 2.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def unbroadcast_reference(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad

In [ ]:
def unbroadcast(grad, shape):
    """Sum grad over the axes that broadcasting stretched, so that the result has exactly `shape`."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
unbroadcast = practice("unbroadcast", unbroadcast, unbroadcast_reference, cases=[
    (upstream_small, (3,)),                         # a bias reused for every row
    (np.arange(12.0).reshape(3, 4), (1, 4)),        # a size-1 axis, kept with keepdims
    (np.arange(24.0).reshape(2, 3, 4), (3, 1)),     # both rules at once
    (np.arange(6.0).reshape(2, 3), ()),             # edge case: a scalar parent
])

**Expected behaviour.** The bias gradient is `[5, 7, 9]`, the column sums. For shape `(1, 4)` you get the column sums `[[12, 15, 18, 21]]`, kept as one row. For `(3, 1)` the leading axis of size 2 is summed away first, then the last axis is summed and kept. For a scalar parent everything is summed into one number, 15. A gradient whose shape already matches comes back unchanged.

## 5 · The backward pass

Every tensor produced by an operation points to its parents, so a loss sits at the end of a **graph** that leads back to the weights and the inputs. Before a node sends gradients to its parents, its own upstream gradient must be complete, which means that every node that used it must already have been processed. Visiting the nodes in reverse **topological order** guarantees exactly that:

- a topological order lists every node *after* all of its parents;
- walking that list backwards therefore visits each node after every node that used it.

A depth-first search produces such an order: visit a node's parents first, then append the node. A set of the ids of visited nodes makes sure that each node is listed once, even when it is reached along two paths.

In [ ]:
p = Tensor(2.0, requires_grad=True)
q = Tensor(3.0, requires_grad=True)
s = p + q
t = s + p          # p is used twice: directly, and through s
names = {id(p): "p", id(q): "q", id(s): "s", id(t): "t"}
for node in (p, q, s, t):
    print(f"{names[id(node)]}: op = {node.op or 'leaf':4}  parents = {[names[id(parent)] for parent in node._parents]}")

### ✍️ Your turn: `topological_order`

Write `topological_order(root)`:

- **input:** `root`, a `Tensor`, usually the loss;
- **output:** a list containing `root` and every tensor it depends on through `_parents`, each exactly once, with every tensor placed after all of its parents. The root therefore comes last.

Hints: start with `order = []` and `seen = set()`. Inside, define `visit(node)`: if `id(node)` is already in `seen`, do nothing; otherwise add it, visit each parent, and then append `node`. Call `visit(root)` and return `order`.

Several orders are correct for most graphs, so the check does not compare your list with the reference's list. Its probe reports four facts about your list instead: `(length, every node after its parents, no repeats, root last)`. For a correct answer they read `(n, True, True, True)`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def topological_order_reference(root):
    order, seen = [], set()

    def visit(node):
        if id(node) not in seen:
            seen.add(id(node))
            for parent in node._parents:
                visit(parent)
            order.append(node)

    visit(root)
    return order

In [ ]:
def topological_order(root):
    """Every tensor that root depends on, and root itself, each once, parents before children."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
def order_report(order_fn, build):
    """Probe: order a freshly built graph and report (length, after parents, no repeats, root last)."""
    root = build()
    order = order_fn(root)
    position = {id(node): i for i, node in enumerate(order)}
    after_parents = all(id(parent) in position and position[id(parent)] < position[id(node)]
                        for node in order for parent in node._parents)
    return (len(order), after_parents, len(position) == len(order), order[-1] is root)


def chain():          # a + b: three nodes
    return Tensor(1.0, True) + Tensor(2.0, True)


def reused():         # s = p + q, then s + p: four nodes, and p is reached twice
    p, q = Tensor(2.0, True), Tensor(3.0, True)
    return (p + q) + p


def wider():          # ((a + b) + (b + c)) + a: seven nodes
    a, b, c = Tensor(1.0, True), Tensor(2.0, True), Tensor(3.0, True)
    return ((a + b) + (b + c)) + a


topological_order = practice_via("topological_order", topological_order, topological_order_reference,
                                 order_report, cases=[(chain,), (reused,), (wider,)])

**Expected behaviour.** The three probes report `(3, True, True, True)`, `(4, True, True, True)` and `(7, True, True, True)`. In the second graph `p` is reached twice, once directly and once through `s`, but it must appear only once; if your list repeats it, the third entry turns `False`.

### Seeding and accumulating

Now walk the order backwards. First **seed** the root: for a scalar loss $\partial L / \partial L = 1$, so `root.grad` becomes an array of ones; a caller may instead pass an upstream gradient for a root that is not a scalar. Then, for each node that has a backward rule and a gradient:

1. call `node._backward(node.grad)` to get one gradient per parent;
2. skip parents that do not require a gradient, such as constants and inputs;
3. bring each gradient back to its parent's shape with `unbroadcast`;
4. **add** it to the parent's `.grad`, or store it if the parent has none yet.

Step 4 is where a value that was used twice collects both contributions. Write it as `parent.grad = parent.grad + g`, which creates a new array, rather than `parent.grad += g`. The rule for addition hands the *same* array to both parents, and an in-place `+=` on one parent's gradient would silently change the other's.

### ✍️ Your turn: `backward_pass`

Write `backward_pass(root, grad=None)`:

- **inputs:** `root`, the tensor to differentiate; `grad`, the upstream gradient for `root`, where `None` means ones (the usual case for a scalar loss);
- **effect:** fills in `.grad` on every tensor that requires one and that `root` depends on. It returns nothing.

Hints: `order = topological_order(root)`. Seed with `np.ones_like(root.data)`, or with `np.asarray(grad, dtype=np.float64)`. Loop over `reversed(order)`, skipping nodes whose `_backward` or `grad` is `None`. `zip(node._parents, node._backward(node.grad))` pairs each parent with its gradient; convert each gradient with `np.asarray(g, dtype=np.float64)` before `unbroadcast`.

The probe builds small graphs, runs your function, and returns the leaf gradients together with whether a constant was correctly left without one.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def backward_pass_reference(root, grad=None):
    order = topological_order_reference(root)
    root.grad = np.ones_like(root.data) if grad is None else np.asarray(grad, dtype=np.float64)
    for node in reversed(order):
        if node._backward is None or node.grad is None:
            continue
        for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
            if not parent.requires_grad or parent_grad is None:
                continue
            parent_grad = unbroadcast_reference(np.asarray(parent_grad, dtype=np.float64), parent.shape)
            parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad

In [ ]:
def backward_pass(root, grad=None):
    """Reverse-mode differentiation: fill in .grad for every tensor that root depends on."""
    # ✍️ Your code here (about ten lines)
    raise NotImplementedError

In [ ]:
def leaf_gradients(backward_fn, build):
    """Probe: build a fresh graph, run backward_fn on it, and return the leaf gradients as plain arrays."""
    root, upstream, leaves, constant = build()
    backward_fn(root, upstream)
    return (*[leaf.grad for leaf in leaves], constant is None or constant.grad is None)


def scalar_loss():       # seeded with ones: both gradients are 1
    a, b = Tensor(2.0, True), Tensor(-3.0, True)
    return a + b, None, [a, b], None


def used_twice():        # (a + a) + b: a collects the upstream gradient twice
    a, b = Tensor([1.0, 2.0, 3.0], True), Tensor([0.5, 0.5, 0.5], True)
    return (a + a) + b, np.array([1.0, -2.0, 0.5]), [a, b], None


def broadcast_bias():    # x (2, 3) + bias (3,): the bias collects the column sums
    x, bias = Tensor(np.ones((2, 3)), True), Tensor(np.zeros(3), True)
    return x + bias, np.arange(6.0).reshape(2, 3), [x, bias], None


def with_constant():     # h = a + 1.0, then h + h: the constant gets no gradient
    a, one = Tensor([1.0, -1.0], True), Tensor(1.0)
    h = a + one
    return h + h, np.array([3.0, 4.0]), [a], one


backward_pass = practice_via("backward_pass", backward_pass, backward_pass_reference, leaf_gradients,
                             cases=[(scalar_loss,), (used_twice,), (broadcast_bias,), (with_constant,)])

**Expected behaviour.** For `a + b` both gradients are 1. In `(a + a) + b`, `a` collects the upstream gradient twice, `[2, -4, 1]`, while `b` receives it once. The bias collects the column sums `[3, 5, 7]` and `x` receives the upstream gradient unchanged. In the last graph `a` receives `[6, 8]`, twice the upstream gradient, and the constant keeps `grad = None`. From now on, `tensor.backward()` runs your function.

### 🧪 Try it: gradients accumulate across calls, too

`backward_pass` adds to whatever `.grad` already holds. That is exactly what a training loop needs when it *deliberately* sums gradients over several small batches, and exactly what breaks training when a loop forgets to clear gradients between updates. Run the cell, then change `calls` and run it again.

In [ ]:
calls = 2  # 🧪 try 1 and 3
w = Tensor([1.0, 2.0], requires_grad=True)
for _ in range(calls):
    (w + w).backward(np.ones(2))    # d(w + w)/dw = 2 for each entry
print(f"after {calls} backward call(s), w.grad = {w.grad}")
w.grad = None                       # clearing, which is what zero_grad does for every parameter
(w + w).backward(np.ones(2))
print("after clearing and one more call:", w.grad)

## 6 · Your first complete operation: multiplication

For elementwise multiplication $y = a \odot b$, each output entry depends on one entry of each input: $\partial y_i / \partial a_i = b_i$ and $\partial y_i / \partial b_i = a_i$. The local rule is therefore

$$\bar a = \bar y \odot b, \qquad \bar b = \bar y \odot a .$$

If `b` was broadcast, $\bar y \odot a$ still has the broadcast shape, and your `backward_pass` repairs it with `unbroadcast`; the rule itself never needs to think about shapes. Compare with `tensor_add` in section 3: an operation computes its output from the parents' `.data` and hands `_result` four things, namely the output, the parents, the rule and a name.

In [ ]:
a_values, b_values = np.array([1.0, 2.0, 3.0]), np.array([4.0, 5.0, 6.0])
upstream = np.ones(3)
print("a * b =", a_values * b_values)
print("with an upstream gradient of ones, a's gradient is b:", upstream * b_values, "and b's gradient is a:", upstream * a_values)

### ✍️ Your turn: `tensor_mul`

Write `tensor_mul(self, other)`, which becomes `Tensor.__mul__`:

- **inputs:** `self`, a `Tensor`; `other`, a `Tensor`, a number or an array (the first line lifts it);
- **output:** a new `Tensor` holding `self.data * other.data`, whose rule maps `g` to `(g * other.data, g * self.data)`, with the op name `"mul"`.

Hint: mirror `tensor_add`. The body is `return self._result(...)` with a `lambda g: (..., ...)`.

The probe runs your operation on fresh tensors, sends back a fixed upstream gradient, and compares the output and both input gradients with the reference.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def tensor_mul_reference(self, other):
    other = Tensor.lift(other)
    return self._result(self.data * other.data, (self, other),
                        lambda g: (g * other.data, g * self.data), "mul")

In [ ]:
def tensor_mul(self, other):
    """Elementwise self * other, with broadcasting. The rule sends (g * other, g * self) back."""
    other = Tensor.lift(other)
    # ✍️ Your code here (one line: return self._result(...))
    raise NotImplementedError

In [ ]:
def forward_and_grads(op, arrays, kwargs=None):
    """Probe: run op on fresh tensors, send back a fixed upstream gradient,
    and return the output followed by each input's gradient, as plain arrays."""
    leaves = [Tensor(np.array(array, dtype=float), requires_grad=True) for array in arrays]
    out = op(*leaves, **(kwargs or {}))
    out.backward(np.cos(np.arange(out.data.size) + 1.0).reshape(out.shape))
    return (out.data, *[leaf.grad for leaf in leaves])


mul_rng = np.random.default_rng(2)
tensor_mul = practice_via("tensor_mul", tensor_mul, tensor_mul_reference, forward_and_grads, cases=[
    ([mul_rng.normal(size=(2, 3)), mul_rng.normal(size=(2, 3))],),
    ([mul_rng.normal(size=(2, 3)), mul_rng.normal(size=(3,))],),     # broadcasting
    ([np.array(2.0), mul_rng.normal(size=(4,))],),                     # edge case: a scalar times a vector
])
Tensor.__mul__ = Tensor.__rmul__ = tensor_mul

**Expected behaviour.** ✅ once your rule matches. In the broadcasting case the probe compares the gradient of the `(3,)` input, which your `backward_pass` has already summed over the rows; in the scalar case, the scalar's gradient is the sum of $\bar y \odot b$. The check cell also attached your function to the class as `*`, and the next cell gradient-checks it against central differences.

In [ ]:
error = gradient_check(lambda a, b: a * b, mul_rng.normal(size=(3, 4)), mul_rng.normal(size=(4,)))
print(f"a * b with broadcasting: relative error {error:.1e}")
assert error < 1e-6, "the rule for multiplication disagrees with central differences"

### The same pattern, many more times

Negation, subtraction, division and powers, and the elementwise functions `exp`, `log`, `tanh` and `sigmoid`, all follow the pattern you have just typed: compute the output, then give each parent $\bar y$ times its local derivative. Read the rules below and match each one to its derivative:

$$\frac{d}{dx} e^x = e^x, \quad \frac{d}{dx}\log x = \frac1x, \quad \frac{d}{dx}\tanh x = 1 - \tanh^2 x, \quad \frac{d}{dx}\sigma(x) = \sigma(x)\bigl(1 - \sigma(x)\bigr), \quad \frac{\partial}{\partial a}\frac{a}{b} = \frac1b, \quad \frac{\partial}{\partial b}\frac{a}{b} = -\frac{a}{b^2}.$$

Subtraction needs no rule of its own, because it is the addition of a negation. The sigmoid is computed as $\tfrac12\bigl(1 + \tanh(x/2)\bigr)$, which is the same function but never overflows. The cell attaches every operation and gradient-checks it.

In [ ]:
def tensor_neg(self):
    return self._result(-self.data, (self,), lambda g: (-g,), "neg")


def tensor_sub(self, other):
    return self + (-Tensor.lift(other))


def tensor_rsub(self, other):
    return Tensor.lift(other) + (-self)


def tensor_truediv(self, other):
    other = Tensor.lift(other)
    return self._result(self.data / other.data, (self, other),
                        lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")


def tensor_rtruediv(self, other):
    return Tensor.lift(other) / self


def tensor_pow(self, exponent):
    assert isinstance(exponent, (int, float)), "only constant exponents"
    return self._result(self.data ** exponent, (self,),
                        lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")


def tensor_exp(self):
    out = np.exp(self.data)
    return self._result(out, (self,), lambda g: (g * out,), "exp")


def tensor_log(self):
    return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")


def tensor_tanh(self):
    out = np.tanh(self.data)
    return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")


def tensor_sigmoid(self):
    out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
    return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")


Tensor.__neg__, Tensor.__sub__, Tensor.__rsub__ = tensor_neg, tensor_sub, tensor_rsub
Tensor.__truediv__, Tensor.__rtruediv__, Tensor.__pow__ = tensor_truediv, tensor_rtruediv, tensor_pow
Tensor.exp, Tensor.log, Tensor.tanh, Tensor.sigmoid = tensor_exp, tensor_log, tensor_tanh, tensor_sigmoid

op_rng = np.random.default_rng(3)
A_pos, B_pos = op_rng.uniform(0.5, 2.0, (3, 4)), op_rng.uniform(0.5, 2.0, (4,))
for name, op, arrays in [("a - b", lambda a, b: a - b, (A_pos, B_pos)),
                         ("1 - a", lambda a: 1.0 - a, (A_pos,)),
                         ("a / b", lambda a, b: a / b, (A_pos, B_pos)),
                         ("2 / a", lambda a: 2.0 / a, (A_pos,)),
                         ("a ** 3", lambda a: a ** 3, (A_pos,)),
                         ("exp(a)", lambda a: a.exp(), (A_pos,)),
                         ("log(a)", lambda a: a.log(), (A_pos,)),
                         ("tanh(a)", lambda a: a.tanh(), (A_pos - 1.0,)),
                         ("sigmoid(a)", lambda a: a.sigmoid(), (A_pos - 1.0,))]:
    error = gradient_check(op, *arrays)
    print(f"{name:11} relative error {error:.1e}")
    assert error < 1e-6, f"the rule for {name} disagrees with central differences"

## 7 · Matrix multiplication

For $Y = AB$ with $A$ of shape $(n, k)$ and $B$ of shape $(k, m)$, each entry is $Y_{ij} = \sum_r A_{ir} B_{rj}$. A single $A_{ir}$ affects a whole row of $Y$, so its gradient sums over that row: $\bar A_{ir} = \sum_j \bar Y_{ij} B_{rj}$. In matrix form, and in the same way for $B$:

$$\bar A = \bar Y B^\top \quad (n, m)(m, k) = (n, k), \qquad \bar B = A^\top \bar Y \quad (k, n)(n, m) = (k, m).$$

You can rebuild both formulas from the shapes alone: each gradient must have its parent's shape, and there is only one way to multiply $\bar Y$ by the other matrix to get it.

For a **batch** of matrices, such as `A` of shape `(batch, n, k)` times a weight `B` of shape `(k, m)`, transpose only the last two axes, with `np.swapaxes(x, -1, -2)`; `.T` would reverse every axis. The formula for $\bar B$ then gives one `(k, m)` gradient per batch entry, shape `(batch, k, m)`, and `unbroadcast` adds them up, because the same weights served every entry.

In [ ]:
A_mm, B_mm, G_mm = np.ones((5, 3)), np.ones((3, 2)), np.ones((5, 2))    # G has the shape of A @ B
print("G @ B.T has shape", (G_mm @ B_mm.T).shape, "which is A's shape", A_mm.shape)
print("A.T @ G has shape", (A_mm.T @ G_mm).shape, "which is B's shape", B_mm.shape)
print("for a batch (4, 5, 3): swapaxes gives", np.swapaxes(np.ones((4, 5, 3)), -1, -2).shape,
      "but .T gives", np.ones((4, 5, 3)).T.shape)

### ✍️ Your turn: `matmul_backward`

Write `matmul_backward(g, a, b)`:

- **inputs:** `g`, the upstream gradient, with the shape of `a @ b`; `a` and `b`, the parents' arrays, each with at least two dimensions;
- **output:** the tuple `(grad_a, grad_b)`, equal to $g\,b^\top$ and $a^\top g$, where $^\top$ swaps the last two axes.

Hint: `np.swapaxes(b, -1, -2)`. Do not sum over the batch yourself; `unbroadcast` does that in the backward pass.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def matmul_backward_reference(g, a, b):
    return g @ np.swapaxes(b, -1, -2), np.swapaxes(a, -1, -2) @ g

In [ ]:
def matmul_backward(g, a, b):
    """Gradients of a @ b, where g has the shape of a @ b. Returns (g @ b^T, a^T @ g) over the last two axes."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
mm_rng = np.random.default_rng(4)
matmul_backward = practice("matmul_backward", matmul_backward, matmul_backward_reference, cases=[
    (mm_rng.normal(size=(5, 2)), mm_rng.normal(size=(5, 3)), mm_rng.normal(size=(3, 2))),
    (mm_rng.normal(size=(4, 5, 2)), mm_rng.normal(size=(4, 5, 3)), mm_rng.normal(size=(3, 2))),  # a batch
    (mm_rng.normal(size=(1, 1)), mm_rng.normal(size=(1, 3)), mm_rng.normal(size=(3, 1))),        # edge case: one number out
])

**Expected behaviour.** Both gradients come back with their parents' shapes. In the batched case your `grad_b` has shape `(4, 3, 2)`, one gradient per batch entry; the backward pass will sum it to `(3, 2)`. The next cell wires your rule into `Tensor.__matmul__`. The method looks `matmul_backward` up by name each time it runs, so it always uses the version that passed the check.

In [ ]:
def tensor_matmul(self, other):
    other = Tensor.lift(other)
    return self._result(self.data @ other.data, (self, other),
                        lambda g: matmul_backward(g, self.data, other.data), "matmul")


Tensor.__matmul__ = tensor_matmul
for name, shapes in [("(5, 3) @ (3, 2)", [(5, 3), (3, 2)]), ("(4, 5, 3) @ (3, 2)", [(4, 5, 3), (3, 2)])]:
    error = gradient_check(lambda a, b: a @ b, *[mm_rng.normal(size=shape) for shape in shapes])
    print(f"{name:20} relative error {error:.1e}")
    assert error < 1e-6, f"the matmul rule disagrees with central differences for {name}"

## 8 · ReLU

$\operatorname{relu}(x) = \max(x, 0)$ passes positive values through and sets the rest to zero, so its derivative is 1 where $x > 0$ and 0 where $x < 0$. At exactly $x = 0$ the function has a kink; like most libraries, we use 0 there. The backward rule is a mask:

$$\bar x = \bar y \odot [x > 0].$$

A unit whose input is negative for every example passes no gradient at all: a "dead" ReLU. That is one reason why weights are initialised with care.

In [ ]:
x_relu = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
print("relu(x) =", np.maximum(x_relu, 0.0))
print("x > 0   =", x_relu > 0, "-> as numbers", (x_relu > 0).astype(float))

### ✍️ Your turn: `relu_backward`

Write `relu_backward(g, x)`:

- **inputs:** `g`, the upstream gradient; `x`, the input of the ReLU, with the same shape;
- **output:** `g` where `x > 0`, and 0 elsewhere, including where `x == 0`.

Hint: multiplying by a boolean array treats `True` as 1 and `False` as 0.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def relu_backward_reference(g, x):
    return g * (x > 0)

In [ ]:
def relu_backward(g, x):
    """Gradient through max(x, 0): g where x > 0, otherwise 0."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
relu_rng = np.random.default_rng(5)
relu_backward = practice("relu_backward", relu_backward, relu_backward_reference, cases=[
    (np.ones(5), x_relu),                                          # includes x == 0
    (relu_rng.normal(size=(3, 4)), relu_rng.normal(size=(3, 4))),
    (relu_rng.normal(size=(2, 2)), -np.ones((2, 2))),              # edge case: every unit is off
])

In [ ]:
def tensor_relu(self):
    return self._result(np.maximum(self.data, 0.0), (self,), lambda g: (relu_backward(g, self.data),), "relu")


Tensor.relu = tensor_relu
away_from_kink = relu_rng.choice([-1.0, 1.0], size=(3, 4)) * relu_rng.uniform(0.1, 2.0, (3, 4))
error = gradient_check(lambda x: x.relu(), away_from_kink)
print(f"relu: relative error {error:.1e}")
assert error < 1e-6, "the ReLU rule disagrees with central differences"

**Expected behaviour.** The first case returns `[0, 0, 0, 1, 1]`: the entry at exactly zero gets no gradient. In the third case every input is negative, so nothing flows back. The gradient check uses inputs at least 0.1 away from zero, because a central difference that straddles the kink measures the average of two slopes, and that would look like an error.

## 9 · Sums and means

`x.sum()` adds every entry, so each entry's derivative is 1: the upstream gradient, a single number, is copied to every position of $x$. With `axis=0` each column becomes one output, and every entry of a column receives that column's upstream value. In general the backward rule **broadcasts the upstream gradient back to the input's shape**. One detail matters: `sum(axis=0)` without `keepdims` removes the axis, so `g` must get it back with `np.expand_dims(g, axis)` before broadcasting. With `keepdims=True` the axis is still there.

A mean is a sum divided by a count, so it needs no new rule: build it from `sum` and division, and its gradient follows automatically.

In [ ]:
x_sum = np.arange(6.0).reshape(2, 3)
g_cols = np.array([10.0, 20.0, 30.0])         # the upstream gradient of x.sum(axis=0), shape (3,)
print("expand_dims(g, 0) has shape", np.expand_dims(g_cols, 0).shape)
print("broadcast back to x's shape:\n", np.broadcast_to(np.expand_dims(g_cols, 0), x_sum.shape))

### ✍️ Your turn: `sum_backward`

Write `sum_backward(g, shape, axis, keepdims)`:

- **inputs:** `g`, the upstream gradient of `x.sum(axis=axis, keepdims=keepdims)`; `shape`, the shape of `x`; `axis`, which is `None`, an int or a tuple of ints; `keepdims`, a bool;
- **output:** an array of shape `shape` in which every entry holds the upstream value of the sum it contributed to.

Hints: if `axis is not None and not keepdims`, restore the removed axes with `np.expand_dims(g, axis)`, which also accepts a tuple. Then return `np.broadcast_to(g, shape).copy()`. The `.copy()` matters, because `broadcast_to` returns a read-only view whose entries share memory.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def sum_backward_reference(g, shape, axis, keepdims):
    if axis is not None and not keepdims:
        g = np.expand_dims(g, axis)
    return np.broadcast_to(g, shape).copy()

In [ ]:
def sum_backward(g, shape, axis, keepdims):
    """Broadcast the upstream gradient of a sum back to the input's shape."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
sum_backward = practice("sum_backward", sum_backward, sum_backward_reference, cases=[
    (np.array(2.0), (2, 3), None, False),                       # the whole sum: one number, spread everywhere
    (g_cols, (2, 3), 0, False),                                 # column sums
    (np.array([[1.0], [2.0]]), (2, 3), 1, True),                # row sums kept as a column
    (np.array([1.0, -1.0, 2.0]), (2, 3, 4), (0, 2), False),     # edge case: two axes at once
])

In [ ]:
def tensor_sum(self, axis=None, keepdims=False):
    return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,),
                        lambda g: (sum_backward(g, self.shape, axis, keepdims),), "sum")


Tensor.sum = tensor_sum
sum_rng = np.random.default_rng(6)
for kwargs in [{}, {"axis": 0}, {"axis": 1, "keepdims": True}]:
    error = gradient_check(lambda x: x.sum(**kwargs), sum_rng.normal(size=(3, 4)))
    print(f"sum{kwargs}: relative error {error:.1e}")
    assert error < 1e-6, "the sum rule disagrees with central differences"

### ✍️ Your turn: `tensor_mean`

Write `tensor_mean(self, axis=None, keepdims=False)`, which becomes `Tensor.mean`:

- **inputs:** as for `sum`;
- **output:** the `Tensor` `self.sum(axis=axis, keepdims=keepdims) / count`, where `count` is the number of entries averaged: `self.data.size` when `axis is None`, and otherwise the product of the sizes of the reduced axes.

Hint: `np.prod([self.shape[a] for a in np.atleast_1d(axis)])` works for an int and for a tuple. You write no backward rule: `sum` and `/` already have theirs.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def tensor_mean_reference(self, axis=None, keepdims=False):
    count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
    return self.sum(axis=axis, keepdims=keepdims) / count

In [ ]:
def tensor_mean(self, axis=None, keepdims=False):
    """Average over axis (over every entry when axis is None), built from sum and division."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
tensor_mean = practice_via("tensor_mean", tensor_mean, tensor_mean_reference, forward_and_grads, cases=[
    ([sum_rng.normal(size=(3, 4))], {}),
    ([sum_rng.normal(size=(3, 4))], {"axis": 0}),
    ([sum_rng.normal(size=(2, 3, 4))], {"axis": (0, 2), "keepdims": True}),   # edge case: two axes, kept
])
Tensor.mean = tensor_mean

**Expected behaviour.** The mean of a `(3, 4)` array sends one twelfth of the upstream gradient to every entry; with `axis=0` each entry receives a third of its column's upstream value. Your function never mentions a gradient, yet the probe's gradients match: composing operations that already have rules is enough.

### Reshaping, transposing and indexing

Three more operations move numbers around without changing them, so their rules move the gradient back. `reshape` reshapes the gradient to the input's shape. `transpose` applies the inverse permutation of the axes. Indexing, as in `x[rows, cols]`, scatters the gradient back into an array of zeros with `x`'s shape. Indexing uses `np.add.at` rather than plain assignment, because the same position can be picked twice (one embedding row used by two tokens, for example), and both picks must add up. Notebook 2 uses indexing to pick each example's score for its true class.

In [ ]:
def tensor_reshape(self, *shape):
    return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")


def tensor_transpose(self, *axes):
    axes = axes or tuple(reversed(range(self.ndim)))
    inverse = np.argsort(axes)
    return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")


def tensor_getitem(self, index):
    def backward(g):
        full = np.zeros_like(self.data)
        np.add.at(full, index, g)  # a position picked twice collects both gradients
        return (full,)

    return self._result(self.data[index], (self,), backward, "index")


Tensor.reshape, Tensor.transpose, Tensor.__getitem__ = tensor_reshape, tensor_transpose, tensor_getitem

shape_rng = np.random.default_rng(7)
picks = (np.array([0, 2, 2]), np.array([1, 0, 0]))   # position (2, 0) is picked twice
for name, op, array in [("reshape", lambda x: x.reshape(4, 6), shape_rng.normal(size=(2, 3, 4))),
                        ("transpose", lambda x: x.transpose(0, 2, 1), shape_rng.normal(size=(2, 3, 4))),
                        ("x.T", lambda x: x.T, shape_rng.normal(size=(3, 4))),
                        ("x[rows, cols]", lambda x: x[picks], shape_rng.normal(size=(3, 2)))]:
    error = gradient_check(op, array)
    print(f"{name:14} relative error {error:.1e}")
    assert error < 1e-6, f"the rule for {name} disagrees with central differences"

## 10 · A stable log-softmax

A classifier turns each row of scores (logits) $z$ into probabilities with the softmax, $p_k = e^{z_k} / \sum_j e^{z_j}$, and its loss uses $\log p_k$. Computing the logarithm directly from the logits is both cheaper and safer:

$$\log p_k = z_k - \log \sum_j e^{z_j}.$$

The danger lies in $e^{z_j}$: `np.exp(1000.0)` is `inf`, and `inf / inf` is `nan`. Subtracting the row maximum $m$ changes nothing mathematically, because $\log \sum_j e^{z_j} = m + \log \sum_j e^{z_j - m}$, but now the largest exponential is $e^0 = 1$ and nothing overflows.

For the backward rule, write $\text{out} = \log p$ for one row. Then $\partial\, \text{out}_i / \partial z_k = [i = k] - p_k$, so for an upstream gradient $\bar y$,

$$\bar z_k = \bar y_k - p_k \sum_i \bar y_i, \qquad \text{where } p = e^{\text{out}}.$$

In [ ]:
z_big = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.log(np.exp(z_big) / np.exp(z_big).sum())
print("naive log-softmax:  ", naive)
print("shifted by the max: ", (z_big - z_big.max()) - np.log(np.exp(z_big - z_big.max()).sum()))

### ✍️ Your turn: `log_softmax_forward`

Write `log_softmax_forward(x, axis=-1)`:

- **inputs:** `x`, an array of logits; `axis`, the axis that holds the classes;
- **output:** an array with the same shape, holding $\log p$ along `axis`, computed with the maximum subtracted first.

Hints: `shifted = x - x.max(axis=axis, keepdims=True)`; then subtract `np.log(np.exp(shifted).sum(axis=axis, keepdims=True))`. `keepdims=True` keeps the shapes broadcastable.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def log_softmax_forward_reference(x, axis=-1):
    shifted = x - x.max(axis=axis, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))

In [ ]:
def log_softmax_forward(x, axis=-1):
    """log(softmax(x)) along axis, computed without overflow."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
ls_rng = np.random.default_rng(8)
log_softmax_forward = practice("log_softmax_forward", log_softmax_forward, log_softmax_forward_reference, cases=[
    (ls_rng.normal(size=(4, 3)),),
    (np.array([[1000.0, 1001.0, 1002.0], [-1000.0, 0.0, 1000.0]]),),   # huge logits must not overflow
    (ls_rng.normal(size=(3, 2)), 0),                                    # classes along axis 0
    (np.array([[5.0], [-5.0]]),),                                       # edge case: one class, so log p = 0
])

### ✍️ Your turn: `log_softmax_backward`

Write `log_softmax_backward(g, out, axis=-1)`:

- **inputs:** `g`, the upstream gradient; `out`, the forward output (log-probabilities), with the same shape; `axis`, the class axis;
- **output:** $\bar z = g - p \sum g$, where $p = e^{\text{out}}$ and the sum runs along `axis`.

Hint: one line, using `g.sum(axis=axis, keepdims=True)`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def log_softmax_backward_reference(g, out, axis=-1):
    return g - np.exp(out) * g.sum(axis=axis, keepdims=True)

In [ ]:
def log_softmax_backward(g, out, axis=-1):
    """Gradient of log_softmax: g minus p times the sum of g along axis, where p = exp(out)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
out_rows = log_softmax_forward_reference(ls_rng.normal(size=(4, 3)))
log_softmax_backward = practice("log_softmax_backward", log_softmax_backward, log_softmax_backward_reference, cases=[
    (ls_rng.normal(size=(4, 3)), out_rows),
    (np.ones((4, 3)), out_rows),                                 # an all-ones upstream gradient
    (ls_rng.normal(size=(3, 2)), log_softmax_forward_reference(ls_rng.normal(size=(3, 2)), 0), 0),
])

**Expected behaviour.** Every row of $\bar z$ sums to zero, whatever the upstream gradient: adding the same constant to every logit in a row leaves the probabilities unchanged, so the loss cannot depend on that direction. The next cell wires both functions into `Tensor.log_softmax` and also adds `softmax`, which later courses use for attention. Even logits of size 50 pass the gradient check.

In [ ]:
def tensor_log_softmax(self, axis=-1):
    out = log_softmax_forward(self.data, axis)
    return self._result(out, (self,), lambda g: (log_softmax_backward(g, out, axis),), "log_softmax")


def tensor_softmax(self, axis=-1):
    shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
    out = shifted / shifted.sum(axis=axis, keepdims=True)
    return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")


Tensor.log_softmax, Tensor.softmax = tensor_log_softmax, tensor_softmax
for name, op, array in [("log_softmax", lambda x: x.log_softmax(axis=-1), ls_rng.normal(size=(4, 3))),
                        ("log_softmax, axis 0", lambda x: x.log_softmax(axis=0), ls_rng.normal(size=(3, 5))),
                        ("log_softmax, large", lambda x: x.log_softmax(axis=-1), 50.0 * ls_rng.normal(size=(4, 3))),
                        ("softmax", lambda x: x.softmax(axis=-1), ls_rng.normal(size=(4, 3)))]:
    error = gradient_check(op, array)
    print(f"{name:20} relative error {error:.1e}")
    assert error < 1e-6, f"the rule for {name} disagrees with central differences"

## 11 · 🧪 Experiments

Your engine now has every operation this course needs. Before the final run, two experiments show what it buys you and how the gradient check protects it. Both use the data defined in the next cell.

The *quadrants* task: a point $(x_1, x_2)$ drawn from a standard normal distribution belongs to class 1 when $x_1 x_2 > 0$, that is, in the top-right or bottom-left quadrant. No straight line separates the two classes, so a model without a hidden layer stays near 50% accuracy. The loss is the mean of $-\log p_y$ over the examples, written here with your log-softmax and indexing; notebook 2 turns it into a function called `cross_entropy`.

In [ ]:
def quadrants(n, rng):
    """n points from a standard normal distribution; class 1 when x1 * x2 > 0."""
    X = rng.normal(size=(n, 2))
    return X, (X[:, 0] * X[:, 1] > 0).astype(int)


def mean_neg_log_likelihood(logits, y):
    return -(logits.log_softmax(axis=-1)[np.arange(len(y)), y]).mean()


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


X_train, y_train = quadrants(400, np.random.default_rng(10))
print("training points:", X_train.shape, "| class 1 share:", y_train.mean())

`parameter` draws weights from $\mathcal N(0, 2/n_\text{in})$, He initialisation, which keeps the scale of ReLU activations roughly constant from layer to layer; `zeros_parameter` starts biases at zero; and `zero_grad` clears gradients before each backward pass.

### 🧪 Experiment 1: what reverse mode saves

The cell times one forward and backward pass against a full numerical gradient of the same loss. Change `hidden_width` and run it again. The numerical gradient needs two forward passes per parameter, so its cost grows with the number of parameters, while the backward pass stays within a small multiple of one forward pass.

In [ ]:
hidden_width = 16  # 🧪 try 64, then 128
cost_rng = np.random.default_rng(13)
V1, c1 = parameter(2, hidden_width, rng=cost_rng), zeros_parameter(hidden_width)
V2, c2 = parameter(hidden_width, 2, rng=cost_rng), zeros_parameter(2)


def cost_loss():
    return mean_neg_log_likelihood((Tensor(X_train) @ V1 + c1).relu() @ V2 + c2, y_train)


start = time.perf_counter()
zero_grad([V1, c1, V2, c2])
cost_loss().backward()
backward_seconds = time.perf_counter() - start
start = time.perf_counter()
numerical = [numerical_grad(cost_loss, p) for p in (V1, c1, V2, c2)]
numerical_seconds = time.perf_counter() - start
worst = max(relative_error(p.grad, n) for p, n in zip((V1, c1, V2, c2), numerical))
print(f"{count_parameters([2, hidden_width, 2])} parameters: forward and backward {1000 * backward_seconds:.1f} ms, "
      f"numerical gradient {1000 * numerical_seconds:.0f} ms, about {numerical_seconds / backward_seconds:.0f} times slower")
print(f"the two gradients agree: worst relative error {worst:.1e}")

**What to notice.** With 82 parameters the numerical gradient is already more than ten times slower (the exact factor depends on your machine), and the gap widens as you add parameters, because the numerical method repeats the whole forward pass for every single weight. The two gradients agree, which is why central differences are useful for *checking* a backward pass but not for training.

### 🧪 Experiment 2: break a rule, and watch the gradient check catch it

The engine's methods look their rules up by name each time they run. The cell temporarily swaps in a broken rule, gradient-checks a small composite expression, and then puts the working rule back. Try each bug and read the relative error.

In [ ]:
bug = "relu_backward lets negative inputs through"  # 🧪 try "log_softmax_backward forgets p" or "none"


def relu_backward_broken(g, x):
    return g * (x < 0)


def log_softmax_backward_broken(g, out, axis=-1):
    return g


working_rules = (relu_backward, log_softmax_backward)
if bug == "relu_backward lets negative inputs through":
    relu_backward = relu_backward_broken
elif bug == "log_softmax_backward forgets p":
    log_softmax_backward = log_softmax_backward_broken
try:
    probe_rng = np.random.default_rng(14)
    error = gradient_check(lambda x, w: (x.relu() @ w).log_softmax(axis=-1),
                           probe_rng.normal(size=(5, 3)), probe_rng.normal(size=(3, 4)))
    print(f"bug = {bug!r}: worst relative error {error:.1e} ->",
          "a rule is wrong" if error > 1e-6 else "every rule agrees with central differences")
finally:
    relu_backward, log_softmax_backward = working_rules   # put the working rules back

**What to notice.** A wrong rule gives a relative error near 1, while correct rules give about $10^{-9}$ or less: the gap is enormous, so the check is a reliable alarm. It cannot tell you *which* rule is wrong in a composite expression, which is why every operation in this notebook was checked on its own first.

## 12 · Put it together: your engine trains a network

Two jobs remain. First, a **gradient-check suite** on composite expressions that exercise several rules at once, including broadcasting, batched matrix products and repeated indices. Second, a **training run**: your engine trains a 2 → 16 → 2 network on the quadrants task with plain gradient descent on all 400 training points, and the trained network is then measured on 1,000 fresh points from the same distribution.

In [ ]:
suite_rng = np.random.default_rng(9)
a_s = Tensor(suite_rng.normal(size=(3, 4)), True)
b_s = Tensor(suite_rng.normal(size=(4,)), True)
c_s = Tensor(suite_rng.normal(size=(4, 2)), True)
x3_s = Tensor(suite_rng.normal(size=(2, 3, 4)), True)
E_s = Tensor(suite_rng.normal(size=(5, 3)), True)
w_s = suite_rng.normal(size=(3, 4))
token_ids = np.array([[0, 2, 2], [4, 0, 1]])
suite = {
    "broadcast add, mul, sum": (lambda: ((a_s + b_s) * a_s).sum(), [a_s, b_s]),
    "sub, div, pow": (lambda: ((a_s - b_s) / (b_s * b_s + 1.0) ** 1.5).sum(), [a_s, b_s]),
    "matmul, tanh": (lambda: (a_s @ c_s).tanh().sum(), [a_s, c_s]),
    "batched matmul, relu, mean": (lambda: (x3_s @ c_s).relu().mean(), [x3_s, c_s]),
    "mean with keepdims": (lambda: (a_s.mean(axis=1, keepdims=True) * a_s).sum(), [a_s]),
    "log_softmax": (lambda: (a_s.log_softmax(axis=-1) * w_s).sum(), [a_s]),
    "repeated embedding rows": (lambda: (E_s[token_ids] ** 2).sum(), [E_s]),
}
for name, (loss_fn, tensors) in suite.items():
    zero_grad(tensors)
    loss_fn().backward()
    worst = max(relative_error(t.grad, numerical_grad(loss_fn, t)) for t in tensors)
    print(f"{name:28} worst relative error {worst:.1e}")
    assert worst < 1e-6, f"{name}: a backward rule disagrees with central differences"

In [ ]:
X_fresh, y_fresh = quadrants(1000, np.random.default_rng(11))    # new points from the same distribution
init_rng = np.random.default_rng(12)
W1, b1 = parameter(2, 16, rng=init_rng), zeros_parameter(16)
W2, b2 = parameter(16, 2, rng=init_rng), zeros_parameter(2)
params = [W1, b1, W2, b2]
assert sum(p.data.size for p in params) == count_parameters([2, 16, 2]) == 82


def forward(X):
    return (Tensor(X) @ W1 + b1).relu() @ W2 + b2


print("nodes in one loss graph:", len(topological_order(mean_neg_log_likelihood(forward(X_train), y_train))))

learning_rate = 0.5
losses = []
for step in range(300):
    loss = mean_neg_log_likelihood(forward(X_train), y_train)
    zero_grad(params)
    loss.backward()
    for p in params:
        p.data -= learning_rate * p.grad          # plain gradient descent on all training points
    losses.append(float(loss.data))

train_accuracy = (forward(X_train).data.argmax(axis=1) == y_train).mean()
fresh_accuracy = (forward(X_fresh).data.argmax(axis=1) == y_fresh).mean()
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}; accuracy {train_accuracy:.1%} on the training points, "
      f"{fresh_accuracy:.1%} on 1,000 fresh points")

In [ ]:
# A last check on the full model: backward() against central differences for every entry of W1.
def small_batch_loss():
    return mean_neg_log_likelihood(forward(X_train[:50]), y_train[:50])


zero_grad(params)
small_batch_loss().backward()
full_model_error = relative_error(W1.grad, numerical_grad(small_batch_loss, W1))
print(f"full-model gradient check on W1: relative error {full_model_error:.1e}")

assert full_model_error < 1e-6, "backward() and central differences disagree on the full model"
assert losses[-1] < 0.5 * losses[0], "the training loss should at least halve"
assert train_accuracy >= 0.9, "the network should fit most of the training points"
assert fresh_accuracy >= 0.85, "on fresh points the network should beat 50% chance by a wide margin"
print("All checks passed: your engine computes correct gradients and trains a network.")

**Expected behaviour.** Every entry of the suite reports a relative error far below $10^{-6}$. The loss graph of one training step has 16 nodes, from the input and the four parameters to the final mean. Training lowers the loss from about 0.78 to about 0.06, the network fits about 99% of the training points, and it classifies about 99% of the fresh points correctly. These numbers describe a two-dimensional teaching task with no label noise; they say nothing about harder data. Notebook 2 adds what real data needs: noisy labels, groups of related rows and a proper split.

## Exercises

Three stretch challenges extend the engine. Each has a check.

### Exercise 1: `layer_norm`, from operations you already have

Layer normalisation rescales each row to mean 0 and variance 1, then applies a learned scale $\gamma$ and shift $\beta$:

$$\operatorname{LN}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}\,\gamma + \beta, \qquad \mu, \sigma^2 \text{ taken along the last axis}.$$

Write `layer_norm(x, gamma, beta, eps=1e-5)` from `mean`, subtraction, `** 2`, `** 0.5`, division, multiplication and addition. You write no backward rule. Transformers in later courses use exactly this function.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def layer_norm_reference(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """Normalise each row of x (along the last axis), then scale by gamma and shift by beta."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
ln_rng = np.random.default_rng(15)
layer_norm = practice_via("layer_norm", layer_norm, layer_norm_reference, forward_and_grads, cases=[
    ([ln_rng.normal(size=(3, 4)), np.ones(4) * 1.1, np.zeros(4) + 0.2],),
    ([ln_rng.normal(size=(2, 3, 5)), ln_rng.normal(size=(5,)), ln_rng.normal(size=(5,))],),
])

### Exercise 2: a new operation, `tensor_leaky_relu`

Leaky ReLU keeps a small slope for negative inputs, so a unit never stops passing gradient completely: $f(x) = x$ for $x > 0$ and $f(x) = s\,x$ otherwise. Write `tensor_leaky_relu(self, slope=0.01)` as a complete operation with its own rule: the local derivative is 1 for $x > 0$ and $s$ otherwise. `np.where(condition, a, b)` is handy for both directions.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def tensor_leaky_relu_reference(self, slope=0.01):
    return self._result(np.where(self.data > 0, self.data, slope * self.data), (self,),
                        lambda g: (g * np.where(self.data > 0, 1.0, slope),), "leaky_relu")

In [ ]:
def tensor_leaky_relu(self, slope=0.01):
    """x where x > 0, else slope * x; the rule multiplies g by 1 or by slope."""
    # ✍️ Your code here (one statement)
    raise NotImplementedError

In [ ]:
leaky_rng = np.random.default_rng(16)
tensor_leaky_relu = practice_via("tensor_leaky_relu", tensor_leaky_relu, tensor_leaky_relu_reference, forward_and_grads, cases=[
    ([leaky_rng.normal(size=(3, 4))], {}),
    ([leaky_rng.normal(size=(3, 4))], {"slope": 0.2}),
    ([-np.ones((2, 2))], {"slope": 0.1}),        # edge case: every input negative
])
Tensor.leaky_relu = tensor_leaky_relu

### Exercise 3: a topological order without recursion

`topological_order` calls `visit` recursively, and Python stops a recursion that is about a thousand calls deep. A network unrolled over a long sequence can build a graph that deep. Write `topological_order_iterative(root)` with an explicit stack of `(node, parents_done)` pairs: when you pop a node whose parents are not done yet, mark it as seen, push it back with `parents_done=True`, then push its unseen parents; when you pop a node whose parents are done, append it to the order. The same probe as before checks the result, so any valid order passes.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def topological_order_iterative_reference(root):
    order, seen, stack = [], set(), [(root, False)]
    while stack:
        node, parents_done = stack.pop()
        if parents_done:
            order.append(node)
        elif id(node) not in seen:
            seen.add(id(node))
            stack.append((node, True))
            stack.extend((parent, False) for parent in node._parents if id(parent) not in seen)
    return order

In [ ]:
def topological_order_iterative(root):
    """The same contract as topological_order, using an explicit stack instead of recursion."""
    # ✍️ Your code here (about ten lines)
    raise NotImplementedError

In [ ]:
topological_order_iterative = practice_via("topological_order_iterative", topological_order_iterative,
                                           topological_order_iterative_reference, order_report,
                                           cases=[(chain,), (reused,), (wider,)])

## Common errors and debugging

- **`ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0 …`.** The inner dimensions differ, for example `(8, 3) @ (2, 16)`. Print both shapes; the last axis of the left operand must equal the first axis of the right one (after any batch axes).
- **A gradient with the wrong shape**, such as `(32, 16)` for a `(16,)` bias. The backward pass did not undo broadcasting; check that `unbroadcast` runs on every parent gradient.
- **Gradients twice as large as expected.** `backward()` ran twice without clearing `.grad` in between. Call `zero_grad(params)` before each backward pass, unless you are accumulating on purpose.
- **Two gradients that change together.** An in-place `parent.grad += g` modified an array that another tensor shares. Write `parent.grad = parent.grad + g`.
- **`nan` or `inf` from a softmax.** The logits were exponentiated without subtracting the row maximum.
- **`.grad` is `None` after `backward()`.** The tensor was created without `requires_grad=True`, its values were copied out through `.data` (which cuts the graph), or it does not affect the loss at all.
- **A gradient check that fails only slightly for ReLU.** A test point lies within `eps` of zero, where the central difference straddles the kink. Move the test points away from zero.
- **`RecursionError` on a very deep graph.** The recursive depth-first search went too deep; use the stack-based order from Exercise 3.
- **`AttributeError: 'Tensor' object has no attribute 'relu'`** (or another operation) after you re-ran the `class Tensor` cell. Re-running it creates a fresh class without the operations that later cells attach. Run the cells below it again, in order, or restart and run everything.

## Summary

- A model is an architecture, a set of weights and an objective. The parameter count follows from the layer sizes, never from the batch size.
- Reverse-mode automatic differentiation records each operation's parents and local rule during the forward pass, then visits the graph in reverse topological order, multiplying by local derivatives and **summing** contributions from every path.
- Broadcasting in the forward pass becomes summation in the backward pass, which `unbroadcast` performs for every operation at once.
- Composite operations such as `mean` and `layer_norm` need no rules of their own.
- A numerically stable log-softmax subtracts the row maximum, and every rule was checked against central differences, which are too slow for training but ideal for testing.

## Next

Notebook 2 builds the data side of training on top of this engine: splitting rows by source so that evaluation means what it claims, fitting a standardiser on training rows only, cutting shuffled mini-batches, and the `cross_entropy` loss.